In [1]:
!pip install -q numpyro
import jax
print("Backend JAX:", jax.default_backend())
print("Dispositivos:", jax.devices())   # debe aparecer un dispositivo CUDA; si dice CPU, activa la GPU

Backend JAX: gpu
Dispositivos: [CudaDevice(id=0)]


In [2]:
!([ -d Bayesian_recommendation_system ] || git clone -q https://github.com/Laura487ortiz/Bayesian_recommendation_system.git) && echo "Repositorio listo"

Repositorio listo


In [10]:
import os, time
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az

USAR_SUBCONJUNTO = False
K           = 5
DRAWS, TUNE, CHAINS = 1000, 1000, 4
TARGET_ACCEPT = 0.95
SEED = 42

In [4]:
# ---------------- Carga de datos ----------------
direc = "Bayesian_recommendation_system/data/processed"
users   = pd.read_csv(os.path.join(direc, "users_clean.csv"))
items   = pd.read_csv(os.path.join(direc, "items_clean.csv"))
ratings = pd.read_csv(os.path.join(direc, "ratings_clean.csv"))

ratings = ratings[
    ratings["user_id"].isin(users["user_id"]) & ratings["movie_id"].isin(items["movie_id"])
].copy()

if USAR_SUBCONJUNTO:
    top_u = ratings["user_id"].value_counts().head(200).index
    top_m = ratings["movie_id"].value_counts().head(300).index
    ratings = ratings[ratings["user_id"].isin(top_u) & ratings["movie_id"].isin(top_m)].copy()

user_codes,  user_ids  = pd.factorize(ratings["user_id"])
movie_codes, movie_ids = pd.factorize(ratings["movie_id"])
ratings["user_idx"]  = user_codes
ratings["movie_idx"] = movie_codes
n_users, n_movies = len(user_ids), len(movie_ids)

rng = np.random.default_rng(SEED)
mask_test = rng.random(len(ratings)) < 0.2
train = ratings[~mask_test].reset_index(drop=True)
test  = ratings[mask_test].reset_index(drop=True)

print(f"Usuarios: {n_users} | Películas: {n_movies} | Ratings: {len(ratings)}")

Usuarios: 998 | Películas: 1681 | Ratings: 100794


In [5]:
# ---------------- Modelo Corregido (v2) ----------------
coords = {
    "user":   user_ids,
    "movie":  movie_ids,
    "factor": [f"f_{i}" for i in range(K)],
}

u_idx = train["user_idx"].to_numpy()
m_idx = train["movie_idx"].to_numpy()
y_obs = train["rating"].to_numpy().astype("float64")

chk = np.random.default_rng(1).choice(len(train), 200, replace=False)

with pm.Model(coords=coords) as modelo:
    mu = pm.Normal("mu", mu=3.0, sigma=1.0)

    sigma_b  = pm.HalfNormal("sigma_b", sigma=1.0)
    sigma_c  = pm.HalfNormal("sigma_c", sigma=1.0)
    sigma_uv = pm.HalfNormal("sigma_uv", sigma=0.5)

    # ZeroSumNormal elimina las crestas con el intercepto
    b_raw = pm.ZeroSumNormal("b_raw", sigma=1.0, dims="user")
    c_raw = pm.ZeroSumNormal("c_raw", sigma=1.0, dims="movie")
    u_raw = pm.ZeroSumNormal("u_raw", sigma=1.0, dims=("factor", "user"))
    v_raw = pm.ZeroSumNormal("v_raw", sigma=1.0, dims=("factor", "movie"))

    u = u_raw * sigma_uv
    v = v_raw * sigma_uv

    # Cálculo optimizado con dims (factor, user)
    interaccion = (u[:, u_idx] * v[:, m_idx]).sum(axis=0)
    rating_est  = mu + b_raw[u_idx] * sigma_b + c_raw[m_idx] * sigma_c + interaccion

    pm.Deterministic("pred_check", rating_est[chk])   # Para diagnosticar convergencia
    sigma_y = pm.HalfNormal("sigma_y", sigma=1.0)
    pm.Normal("rating_obs", mu=rating_est, sigma=sigma_y, observed=y_obs)

In [6]:
# ---------------- Muestreo (NumPyro) ----------------
en_gpu = jax.default_backend() != "cpu"
metodo_cadenas = "vectorized" if en_gpu else "sequential"
print("Backend:", jax.default_backend(), "| chain_method:", metodo_cadenas)

t0 = time.time()
with modelo:
    trace = pm.sample(
        draws=DRAWS,
        tune=TUNE,
        chains=CHAINS,
        target_accept=TARGET_ACCEPT,
        nuts_sampler="numpyro",
        nuts_sampler_kwargs={"chain_method": metodo_cadenas},
        random_seed=SEED,
    )
print(f"Tiempo de muestreo: {(time.time() - t0) / 60:.1f} min")

Backend: gpu | chain_method: vectorized


sample: 100%|██████████| 2000/2000 [11:24<00:00,  2.92it/s]
/usr/local/lib/python3.13/dist-packages/jax/_src/interpreters/mlir.py:1406: UserWarning: Some donated buffers were not usable: float64[4,1000,997], float64[4,1000,1680], float64[4,1000,5,997], float64[4,1000,5,1680].
See an explanation at https://docs.jax.dev/en/latest/faq.html#buffer-donation.
  warnings.warn("Some donated buffers were not usable:"
ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Tiempo de muestreo: 13.7 min


In [7]:
# ---------------- Diagnósticos ----------------
resumen = az.summary(trace, var_names=["mu", "sigma_b", "sigma_c", "sigma_uv", "sigma_y"])
print(resumen)

sesgos_pred = az.summary(trace, var_names=["b_raw", "c_raw", "pred_check"])
print("\nR-hat máx. (sesgos y predicciones):", float(sesgos_pred["r_hat"].max()))
print("ESS bulk mín. (sesgos y predicciones):", float(sesgos_pred["ess_bulk"].min()))
print("Divergencias:", int(trace.sample_stats["diverging"].sum().values))

           mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
mu        3.277  0.010   3.259    3.295        0.0      0.0    1401.0   
sigma_b   0.390  0.011   0.369    0.409        0.0      0.0     911.0   
sigma_c   0.512  0.012   0.488    0.533        0.0      0.0     699.0   
sigma_uv  0.455  0.005   0.446    0.463        0.0      0.0    1174.0   
sigma_y   0.826  0.002   0.822    0.831        0.0      0.0    2944.0   

          ess_tail  r_hat  
mu          2674.0   1.00  
sigma_b     1860.0   1.00  
sigma_c     1369.0   1.00  
sigma_uv    1722.0   1.01  
sigma_y     3299.0   1.00  

R-hat máx. (sesgos y predicciones): 1.01
ESS bulk mín. (sesgos y predicciones): 832.0
Divergencias: 0


In [12]:
post = trace.posterior

def aplanar(nombre):
    x = post[nombre].values
    return x.reshape((-1,) + x.shape[2:])

mu_s, sb_s, sc_s, suv_s = (aplanar(n) for n in ["mu", "sigma_b", "sigma_c", "sigma_uv"])
b_raw_s, c_raw_s = aplanar("b_raw"), aplanar("c_raw")

u_raw_s = aplanar("u_raw").swapaxes(1, 2)
v_raw_s = aplanar("v_raw").swapaxes(1, 2)

def predecir(df, max_draws=500, lote=50):
    ui = df["user_idx"].to_numpy()
    mi = df["movie_idx"].to_numpy()
    sel = np.linspace(0, len(mu_s) - 1, min(len(mu_s), max_draws)).astype(int)
    acum = np.zeros(len(df))
    for i in range(0, len(sel), lote):
        s = sel[i:i + lote]
        inter = np.einsum("snk,snk->sn", u_raw_s[s][:, ui, :], v_raw_s[s][:, mi, :])
        pred = (
            mu_s[s, None]
            + b_raw_s[s][:, ui] * sb_s[s, None]
            + c_raw_s[s][:, mi] * sc_s[s, None]
            + (suv_s[s, None] ** 2) * inter
        )
        acum += pred.sum(axis=0)
    return np.clip(acum / len(sel), 1, 5)

pred_test = predecir(test)
y_test = test["rating"].to_numpy()

rmse = np.sqrt(np.mean((pred_test - y_test) ** 2))
mae  = np.mean(np.abs(pred_test - y_test))
base = train["rating"].mean()
rmse_base = np.sqrt(np.mean((base - y_test) ** 2))

print(f"RMSE modelo:             {rmse:.4f}")
print(f"MAE modelo:              {mae:.4f}")
print(f"RMSE baseline (media):   {rmse_base:.4f}")
print(f"Mejora vs baseline:      {100 * (1 - rmse / rmse_base):.1f}%")

RMSE modelo:             0.9064
MAE modelo:              0.7120
RMSE baseline (media):   1.1284
Mejora vs baseline:      19.7%


In [9]:
# ---------------- PASO 8: Predicción Top-N ----------------
print("Generando Top-N de recomendaciones...")

# Extraemos las medias posteriores de los parámetros
mu_hat = trace.posterior['mu'].mean().item()
b_hat = trace.posterior['b_raw'].mean(dim=['chain', 'draw']).values * trace.posterior['sigma_b'].mean().item()
c_hat = trace.posterior['c_raw'].mean(dim=['chain', 'draw']).values * trace.posterior['sigma_c'].mean().item()

# Los factores están en dims (factor, user). Hacemos la transpuesta para que sea (user, factor)
u_hat = (trace.posterior['u_raw'].mean(dim=['chain', 'draw']).values * trace.posterior['sigma_uv'].mean().item()).T
v_hat = (trace.posterior['v_raw'].mean(dim=['chain', 'draw']).values * trace.posterior['sigma_uv'].mean().item()).T

def recomendar_top_n(user_idx, n=10):
    # Calificación esperada para TODAS las películas para este usuario
    scores = mu_hat + b_hat[user_idx] + c_hat + np.dot(u_hat[user_idx], v_hat.T)

    # Obtener las películas que el usuario ya vio (en train)
    vistas = train[train['user_idx'] == user_idx]['movie_idx'].values
    scores[vistas] = -np.inf  # Ocultar las que ya vio

    # Obtener los índices top N
    top_indices = np.argsort(scores)[-n:][::-1]

    # Mapear de vuelta al ID original de la película
    top_movie_ids = movie_ids[top_indices]
    top_scores = scores[top_indices]

    return top_movie_ids, top_scores

# --- Ejemplo de uso con el primer usuario (user_idx = 0) ---
usuario_ejemplo = 0
id_original_usuario = user_ids[usuario_ejemplo]

top_peliculas, scores = recomendar_top_n(usuario_ejemplo, n=10)

print(f"\nTop 10 recomendaciones para el Usuario ID: {id_original_usuario}")
print("-" * 50)

# Buscar los títulos (asumiendo que la tabla items tiene una columna 'title' o 'movie_title')
# Si tu columna se llama diferente, cámbiala aquí abajo:
col_titulo = 'title' if 'title' in items.columns else 'movie_title'

for i, (m_id, score) in enumerate(zip(top_peliculas, scores), 1):
    # Buscar el título de la película
    info_pelicula = items[items['movie_id'] == m_id]
    if not info_pelicula.empty and col_titulo in info_pelicula.columns:
        titulo = info_pelicula[col_titulo].values[0]
    else:
        titulo = f"Película ID {m_id}"

    print(f"{i}. {titulo} (Score predicho: {score:.2f})")

Generando Top-N de recomendaciones...

Top 10 recomendaciones para el Usuario ID: 196
--------------------------------------------------
1. Shawshank Redemption, The (1994) (Score predicho: 4.64)
2. Schindler's List (1993) (Score predicho: 4.58)
3. 12 Angry Men (1957) (Score predicho: 4.45)
4. Rear Window (1954) (Score predicho: 4.45)
5. Casablanca (1942) (Score predicho: 4.45)
6. Silence of the Lambs, The (1991) (Score predicho: 4.43)
7. Usual Suspects, The (1995) (Score predicho: 4.42)
8. One Flew Over the Cuckoo's Nest (1975) (Score predicho: 4.41)
9. Good Will Hunting (1997) (Score predicho: 4.39)
10. Close Shave, A (1995) (Score predicho: 4.38)
